# 04C — Đọc baseline hoặc experiment theo tên (suite hoặc run exp010)

Read-only: không tạo features, train hoặc chạy lại model trên TEST.
Đặt EXPERIMENT_NAME bằng tên đã dùng trong 04D; để trống để xem baseline cố định.
Protocol v2/v3/v4/v5 chọn checkpoint theo validation Macro-F1; TEST tắt khi tuning.
Bảng, histories, confusion matrices và diagnostics lấy từ experiment đã chọn.
Shortlist chỉ dùng validation; source-level metrics là chỉ số phụ.

Mặc định exp010: một DS-CNN/03A với TRAIN time masking; đối chứng exp007.
Exp009 gộp window ở mục riêng bên dưới. Kiểm tra params đã lưu và so window/source F1 riêng.


## exp009 — Gộp hai window từ dự đoán exp007 (CPU, không train)

Chạy riêng các cell của mục này; không cần chạy các mục so sánh model bên dưới hoặc 04D.
Trong Colab chọn runtime **CPU**, mount Drive và dùng repo mới nhất. Nếu đã có `/content/edge-ai`, pull repo trước và restart kernel khi cần.

Đối chứng cố định: `exp_007_power075 / 03A_ds_cnn_seed42`, chỉ validation. Ghép `(0,1), (2,3), ...` trong cùng clip có timestamp liên tiếp; không ghép qua clip/source. Window cuối lẻ được liệt kê và loại khỏi cả hai phép so sánh. Hai window có 15ms audio chung; tổng thời lượng hỗ trợ là **1,935 giây**.

`DIAGNOSTIC_NAME` là tên thư mục kết quả riêng. Để `SAVE_DIAGNOSTIC=False` xem preview, rồi đổi thành `True` để lưu. Tên đã có bị từ chối; dùng cell đọc lại bên dưới hoặc tên `_retry01`. Không đặt `EXPERIMENT_NAME` của phần training review bên dưới thành exp009: đây là diagnostic, không phải experiment train tám model.


In [3]:
# Each notebook kernel discovers its own runtime checkout.
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
_repo_hint = os.environ.get("EDGEAI_REPO_ROOT")
if _repo_hint and os.name != "nt" and PureWindowsPath(_repo_hint).drive:
    raise RuntimeError("EDGEAI_REPO_ROOT must name the checkout in this runtime, not a Windows drive")
_start = Path(_repo_hint).expanduser().resolve() if _repo_hint else Path.cwd().resolve()
_candidates = [_start, *_start.parents, Path("/content/edge-ai")]
_repo = next((p for p in _candidates if (p / "tools/project_paths.py").is_file()), None)
if _repo is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout and rerun this cell") from None
    _repo = Path(_repo_hint or "/content/edge-ai").expanduser().resolve()
    if _repo.exists():
        raise RuntimeError(f"{_repo} exists but is not an EdgeAI checkout; set EDGEAI_REPO_ROOT to the correct checkout")
    subprocess.run(["git", "clone", "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git", str(_repo)], check=True)
# Cập nhật checkout Colab trước khi import module diagnostic.
try:
    import google.colab
except ImportError:
    _diagnostic_colab = False
else:
    _diagnostic_colab = True
if _diagnostic_colab:
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=_repo, text=True).strip():
        raise RuntimeError("Checkout có thay đổi chưa commit; giữ các thay đổi trước khi pull repo")
    subprocess.run(["git", "pull", "--ff-only"], cwd=_repo, check=True)
sys.path.insert(0, str(_repo / "tools"))
sys.dont_write_bytecode = True
from project_paths import REPO_ROOT, ProjectPaths, COLAB_DATA_ROOT, is_colab_runtime
if REPO_ROOT != _repo.resolve():
    raise RuntimeError("Kernel đã import checkout khác; restart kernel và chạy lại cell này")
os.environ["EDGEAI_REPO_ROOT"] = str(_repo)
os.chdir(_repo)
if is_colab_runtime():
    from google.colab import drive
    if not COLAB_DATA_ROOT.parent.is_dir():
        drive.mount("/content/drive")
PATHS = ProjectPaths.from_env()

from bounded_aggregation import run as run_two_window, load_saved as read_two_window
import pandas as pd
from IPython.display import display

# CHỈNH TÊN VÀ CỜ LƯU Ở ĐÂY; không có cờ training.
DIAGNOSTIC_NAME = "exp_009_two_window_agg"
SAVE_DIAGNOSTIC = True
print("Kết quả dự kiến:", PATHS.experiment(DIAGNOSTIC_NAME))


Kết quả dự kiến: /content/drive/MyDrive/EdgeAI/experiments/exp_009_two_window_agg


In [4]:
diagnostic, diagnostic_summary, diagnostic_classes, diagnostic_path = run_two_window(
    DIAGNOSTIC_NAME, save=SAVE_DIAGNOSTIC)
print("ĐÃ LƯU" if SAVE_DIAGNOSTIC else "PREVIEW — chưa ghi kết quả", diagnostic_path)
print("Số mẫu:", diagnostic["population"])
display(diagnostic_summary)
display(diagnostic_classes)
display(pd.DataFrame(diagnostic["paired_source_bootstrap"]["intervals"]).T)
print("So sánh chính: matched_single_window vs two_window; all_validation_single_window chỉ là tham khảo.")
print("CI bootstrap theo source, chưa bao gồm khác biệt seed hoặc việc đã tuning validation.")
print("Đây là quyết định trên 1,935s âm thanh; chưa chứng minh model một window tốt hơn hoặc kết quả TEST.")


ĐÃ LƯU /content/drive/MyDrive/EdgeAI/experiments/exp_009_two_window_agg
Số mẫu: {'original_windows': 4937, 'matched_windows': 4778, 'pairs': 2389, 'excluded_windows': 159, 'original_sources': 177, 'matched_sources': 177, 'clips': 269}


,endpoint,unit,examples,macro_f1_pct,accuracy_pct
0,all_validation_single_window,window,4937,60.974837,66.254811
1,matched_single_window,window,4778,61.324459,66.617832
2,two_window,pair,2389,64.900967,69.945584


,endpoint,species,precision,recall,f1,support
0,all_validation_single_window,an arabiensis,0.696558,0.738288,0.716816,2220
1,all_validation_single_window,an funestus ss,0.728175,0.655357,0.689850,1120
2,all_validation_single_window,an squamosus,0.384000,0.488136,0.429851,295
3,all_validation_single_window,culex pipiens complex,0.627810,0.579109,0.602477,1302
4,matched_single_window,an arabiensis,0.698774,0.743017,0.720217,2148
5,matched_single_window,an funestus ss,0.734568,0.656250,0.693204,1088
6,matched_single_window,an squamosus,0.387187,0.492908,0.433697,282
7,matched_single_window,culex pipiens complex,0.631126,0.582540,0.605861,1260
8,two_window,an arabiensis,0.716965,0.783054,0.748554,1074
9,two_window,an funestus ss,0.777778,0.681985,0.726738,544


,delta_f1_pp,ci95_low_pp,ci95_high_pp
macro,3.576508,2.222047,4.854830
an arabiensis,2.833702,1.563293,4.103131
an funestus ss,3.353461,1.501135,5.190639
an squamosus,4.790800,0.701613,8.235532
culex pipiens complex,3.328070,1.422954,5.221114


So sánh chính: matched_single_window vs two_window; all_validation_single_window chỉ là tham khảo.
CI bootstrap theo source, chưa bao gồm khác biệt seed hoặc việc đã tuning validation.
Đây là quyết định trên 1,935s âm thanh; chưa chứng minh model một window tốt hơn hoặc kết quả TEST.


### Đọc lại exp009 đã lưu
Chạy sau cell thiết lập phía trên nếu muốn xem lần cũ; không cần chạy lại phép tính hoặc bật cờ lưu.


In [5]:
saved_diagnostic, saved_summary, saved_classes, saved_path = read_two_window(DIAGNOSTIC_NAME)
print("Đã xác thực kết quả lưu:", saved_path)
display(saved_summary)
display(saved_classes)
display(pd.DataFrame(saved_diagnostic["paired_source_bootstrap"]["intervals"]).T)


Đã xác thực kết quả lưu: /content/drive/MyDrive/EdgeAI/experiments/exp_009_two_window_agg


,endpoint,unit,examples,macro_f1_pct,accuracy_pct
0,all_validation_single_window,window,4937,60.974837,66.254811
1,matched_single_window,window,4778,61.324459,66.617832
2,two_window,pair,2389,64.900967,69.945584


,endpoint,species,precision,recall,f1,support
0,all_validation_single_window,an arabiensis,0.696558,0.738288,0.716816,2220
1,all_validation_single_window,an funestus ss,0.728175,0.655357,0.689850,1120
2,all_validation_single_window,an squamosus,0.384000,0.488136,0.429851,295
3,all_validation_single_window,culex pipiens complex,0.627810,0.579109,0.602477,1302
4,matched_single_window,an arabiensis,0.698774,0.743017,0.720217,2148
5,matched_single_window,an funestus ss,0.734568,0.656250,0.693204,1088
6,matched_single_window,an squamosus,0.387187,0.492908,0.433697,282
7,matched_single_window,culex pipiens complex,0.631126,0.582540,0.605861,1260
8,two_window,an arabiensis,0.716965,0.783054,0.748554,1074
9,two_window,an funestus ss,0.777778,0.681985,0.726738,544


,delta_f1_pp,ci95_low_pp,ci95_high_pp
macro,3.576508,2.222047,4.854830
an arabiensis,2.833702,1.563293,4.103131
an funestus ss,3.353461,1.501135,5.190639
an squamosus,4.790800,0.701613,8.235532
culex pipiens complex,3.328070,1.422954,5.221114


In [1]:
# Each notebook kernel discovers its own runtime checkout.
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
_repo_hint = os.environ.get("EDGEAI_REPO_ROOT")
if _repo_hint and os.name != "nt" and PureWindowsPath(_repo_hint).drive:
    raise RuntimeError("EDGEAI_REPO_ROOT must name the checkout in this runtime, not a Windows drive")
_start = Path(_repo_hint).expanduser().resolve() if _repo_hint else Path.cwd().resolve()
_candidates = [_start, *_start.parents, Path("/content/edge-ai")]
_repo = next((p for p in _candidates if (p / "tools/project_paths.py").is_file()), None)
if _repo is None:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout and rerun this cell") from None
    _repo = Path(_repo_hint or "/content/edge-ai").expanduser().resolve()
    if _repo.exists():
        raise RuntimeError(f"{_repo} exists but is not an EdgeAI checkout; set EDGEAI_REPO_ROOT to the correct checkout")
    subprocess.run(["git", "clone", "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git", str(_repo)], check=True)
if not (_repo / "tools/notebook_runtime.py").is_file():
    raise RuntimeError("Checkout is missing notebook_runtime.py. Pull the latest GitHub commit in this runtime, then rerun this cell")
sys.path.insert(0, str(_repo / "tools"))
from notebook_runtime import initialize
PATHS = initialize(_repo)
from project_paths import REPO_ROOT, ProjectPaths, resolve_path, logical_relative
from pathlib import Path
import sys
sys.dont_write_bytecode = True
PROJECT = REPO_ROOT
sys.path.insert(0, str(PROJECT / "tools"))
import importlib
import stage04_data, model_zoo
for module in (stage04_data, model_zoo):
    importlib.reload(module)
import pandas as pd
from IPython.display import display
from model_zoo import FAMILIES, architecture_config, build_model
from stage04_data import BRANCHES, CLASSES, ROOT, prepare_caches, read_json


Repository: /content/edge-ai
Data: /content/drive/MyDrive/EdgeAI
Immutable baseline: /content/drive/MyDrive/EdgeAI/fixed/baseline/stage04
Future results: /content/drive/MyDrive/EdgeAI/experiments/<experiment_id>


In [ ]:
# Để trống: baseline cố định. Nhập tên đã dùng trong 04D: kết quả experiment trên Drive.
EXPERIMENT_NAME = "exp_010_time_mask"

all_complete = False  # Prevent plots from reusing a previous selection after a failed check.
RUNS = None
result_root = (PATHS.experiment(EXPERIMENT_NAME) / "results"
               if EXPERIMENT_NAME else PATHS.BASELINE_ROOT)
print("Đang đọc:", result_root)
if EXPERIMENT_NAME:
    config_path = result_root.parent / "config/experiment.json"
    if not config_path.is_file():
        raise FileNotFoundError(f"Chưa có experiment {EXPERIMENT_NAME} tại {result_root.parent}; chạy 04D hoặc chọn tên lần cũ.")
    from experiment_runner import run_plan
    review_plan = run_plan(read_json(config_path))
else:
    review_plan = [{"frontend": f, "model": m, "run_id": f"{f}_{m}_seed42"}
                   for f in BRANCHES for m in FAMILIES]
status = []
for item in review_plan:
    path = result_root / "runs" / item["run_id"] / "result.json"
    status.append({"Frontend": item["frontend"], "Model": item["model"],
                   "Status": read_json(path)["status"] if path.is_file() else "NOT RUN"})
display(pd.DataFrame(status))
if EXPERIMENT_NAME:
    metadata = read_json(result_root.parent / "config/metadata.json")
    print("Experiment status:", metadata["status"],
          f'{metadata["completed_runs"]}/{metadata["expected_runs"]}')
    display(pd.Series({"experiment_id": metadata["experiment_id"],
                       "git_commit": metadata["git_commit"],
                       "description": metadata["configuration"]["description"],
                       "configuration": metadata["configuration"],
                       "runtime": metadata["runtime"]}, name="Run provenance"))
    from experiment_review import verify_experiment, comparison_summary
    records = verify_experiment(EXPERIMENT_NAME)
    summary, comparison = comparison_summary(records)
else:
    from baseline_review import verify_baseline
    import experiment_training
    records = verify_baseline()
    comparison = pd.DataFrame([experiment_training.comparison_row(r) for r in records])
    summary = read_json(result_root / "comparison.json")
RUNS = result_root / "runs"
all_complete = True
print("Saved training protocol/parameters:", read_json(result_root / "protocol.json")["training_configuration"])
print("TEST evaluated:", all(r.get("test_evaluated", True) for r in records))
display(comparison)
print("Runs đã xác thực:", [r["run_id"] for r in records])
if len(records) > 1:
    print("Provisional Stage-05 shortlist (validation only):", summary["stage05_shortlist"])
else:
    print("Đây là một ablation; dùng cell đối chiếu exp007 bên dưới, không xếp hạng giữa các model.")


## Saved training histories and confusion matrices

TRAIN history là online metrics trong lúc cập nhật trọng số; validation chạy eval.
V2 bổ sung TRAIN eval tại checkpoint đã chọn để so sánh công bằng hơn.
Đường đánh dấu thể hiện epoch chọn theo F1 và epoch tốt nhất theo loss.
Confusion matrices có hàng là nhãn thật, cột là nhãn dự đoán; luôn giữ đủ bốn lớp.
V2 hiển thị validation và TRAIN eval; TEST chỉ hiện nếu experiment thực sự đã đánh giá TEST.


In [ ]:
if all_complete:
    import matplotlib.pyplot as plt
    labels_short = ["Ara", "Fun", "Squ", "Cpx"]
    for record in records:
        frontend, family = record["frontend"], record["model_family"]
        directory = RUNS / record["run_id"]
        result = read_json(directory / "result.json")
        history = read_json(directory / "history.json")
        fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
        epochs = [h["epoch"] for h in history]
        for split in ("train", "validation"):
            axes[0].plot(epochs, [h[split]["loss"] for h in history], label=split)
            axes[1].plot(epochs, [h[split]["macro_f1"] for h in history], label=split)
        if "train_eval_metrics" in result:
            axes[0].scatter([result["best_epoch"]], [result["train_eval_metrics"]["loss"]], label="TRAIN eval checkpoint", marker="x")
            axes[1].scatter([result["best_epoch"]], [result["train_eval_metrics"]["macro_f1"]], label="TRAIN eval checkpoint", marker="x")
        for ax in axes[:2]:
            ax.axvline(result["best_epoch"], color="green", linestyle="--", label="selected epoch")
            if "checkpoint_candidates" in result:
                ax.axvline(result["checkpoint_candidates"]["minimum_loss"]["epoch"], color="gray", linestyle=":", label="min-loss epoch")
            ax.legend(fontsize=7)
        axes[0].set(title="Weighted loss", xlabel="Epoch")
        axes[1].set(title="Macro-F1", xlabel="Epoch")
        secondary = result.get("final_test_metrics") or result.get("train_eval_metrics")
        secondary_title = "TEST" if result.get("final_test_metrics") else "TRAIN eval"
        for ax, metrics, title in ((axes[2], result["selected_validation_metrics"], "Validation"),
                                   (axes[3], secondary, secondary_title)):
            if metrics is None:
                ax.set_visible(False)
                continue
            cm = metrics["confusion_matrix"]
            ax.imshow(cm, cmap="Blues")
            for i in range(4):
                for j in range(4):
                    ax.text(j, i, cm[i][j], ha="center", va="center")
            ax.set(title=title + " window confusion", xlabel="Predicted", ylabel="True",
                   xticks=range(4), yticks=range(4), xticklabels=labels_short, yticklabels=labels_short)
        fig.suptitle(f"{EXPERIMENT_NAME or 'fixed baseline'} / {frontend} / {family}; selected epoch {result['best_epoch']}")
        fig.tight_layout(); plt.show()
        print("Validation per-class:")
        display(pd.DataFrame(result["selected_validation_metrics"]["per_class"]).T)
        if "validation_source_metrics" in result:
            display(pd.Series(result["validation_source_metrics"], name="Validation source evaluation"))
        if result.get("final_test_metrics") is not None:
            print("TEST per-class (descriptive, not for tuning):")
            display(pd.DataFrame(result["final_test_metrics"]["per_class"]).T)
            display(pd.Series(result["source_test_metrics"], name="TEST source evaluation"))
        else:
            print("TEST chưa được đánh giá trong experiment này; dùng validation để tuning.")
        if (directory / "class_weights.json").is_file():
            print("TRAIN class-weight policy and effective weights:")
            display(pd.Series(read_json(directory / "class_weights.json")))
        if (directory / "diagnostics.json").is_file():
            diagnostics = read_json(directory / "diagnostics.json")
            print("Validation species × acquisition method (support + recall):")
            display(pd.DataFrame(diagnostics["validation_subgroups"]))
            print("BatchNorm running statistics tại checkpoint được chọn:")
            display(pd.DataFrame(diagnostics["batchnorm_selected_checkpoint"]).T)


## So sánh validation với exp007

Exp010 đối chiếu đúng run DS-CNN/03A của exp007, cùng split/cache/model và trọng số CE.
Các protocol suite cũ vẫn yêu cầu tập run giống nhau. TEST tắt; F1 source khác F1 window.
Tham số augmentation, TRAIN eval không che và validation được lưu riêng trong protocol/results.


In [ ]:
REFERENCE_EXPERIMENT_NAME = ""  # Để trống: dùng reference_experiment_id từ metadata.
ADDITIONAL_REFERENCE_EXPERIMENT_NAME = ""  # Để trống nếu chỉ muốn một đối chứng.
if all_complete and EXPERIMENT_NAME:
    from experiment_review import compare_validation_to_reference
    primary = REFERENCE_EXPERIMENT_NAME or metadata["configuration"].get("reference_experiment_id")
    references = list(dict.fromkeys(name for name in (primary, ADDITIONAL_REFERENCE_EXPERIMENT_NAME)
                                    if name and name != EXPERIMENT_NAME))
    for reference_name in references:
        reference_path = PATHS.experiment(reference_name)
        if (reference_path / "config/metadata.json").is_file():
            print("Validation comparison:", EXPERIMENT_NAME, "vs", reference_name)
            display(compare_validation_to_reference(EXPERIMENT_NAME, reference_name))
        else:
            print("Chưa tìm thấy reference:", reference_path, "; kết quả hiện tại vẫn được xác thực riêng.")
    if not references:
        print("Nhập tên experiment đối chứng khác với experiment đang xem.")


## Limits before Stage 05

No Edge winner is declared. Float32 checkpoint size is not INT8 size, RAM, Flash or latency.
This one-seed comparison makes no claims of superiority to published literature. Multiple windows may
share one recording; source-name isolation does not prove biological/session/domain independence.
The frozen acquisition-method confounding remains. Read [Stage 04 methodology](../../docs/STAGE04.md).
